# Лабораторная работа 05. Scikit-learn: первая модель оценки цены

**Курс:** «Анализ данных и искусственный интеллект», 1 курс бакалавриата.

Мы уже подготовили и изучили данные об автомобилях. Теперь хотим по характеристикам автомобиля получить оценку цены. Не будем начинать с большого цикла моделей: пройдём один эксперимент маленькими шагами.

**После работы:** вы сможете объяснить, что такое `X` и `y`, зачем разделять данные, что делают `fit` и `predict`, как прочитать MAE и как применить масштабирование без подглядывания в проверочные данные.

**Нужно знать:** основы Python и Pandas из работы 02. Материал работ 03–04 помогает понимать таблицы и графики.

**Основной маршрут:** разделы 1–8 и итоговый отчёт. Пункт про MAPE/R² — дополнительный.
**Темп:** выполняйте одну ячейку, изучайте результат, отвечайте на вопрос после неё и только затем переходите дальше. Если пара длится 90 минут, разделы 7–8 можно завершить дома. Для спокойного разбора всей работы разумно выделить две пары; это ориентир для планирования занятия, а не обязательный срок.


## 1. Что именно мы просим модель сделать

Один объект — автомобиль в одной строке. Его признаки — год выпуска, пробег, объём двигателя и условный среднегодовой пробег. Правильный ответ для обучения — указанная цена в евро.

| Термин | В нашей задаче |
|---|---|
| Объект | Одна строка с автомобилем |
| Признаки | Известные характеристики автомобиля |
| Целевая переменная | Цена `Price` |
| Обучение | Настройка модели по признакам и известным ценам |
| Предсказание | Получение оценки цены по признакам |
| Регрессия | Предсказание численного значения |

Обученная модель не узнаёт состояние машины и фактическую цену сделки. Она ищет закономерности в доступных признаках и ценах предложений.

**Остановитесь:** какая колонка здесь является ответом? Можно ли передать её модели как входной признак при оценке неизвестной цены?


### Подключаем библиотеки

`import` подключает библиотеку. Через `from ... import ...` берём конкретный инструмент. Пока подключаем только работу с таблицами, графиками и разделение данных.

Установка при необходимости: `pip install pandas numpy matplotlib scikit-learn joblib`. После установки перезапустите ядро. Установочный пакет называется `scikit-learn`, а импорт — `sklearn`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
import sklearn

print('Версия scikit-learn:', sklearn.__version__)


Появился номер версии. Модель пока не создана и не обучена.

**Остановитесь:** чем подключение библиотеки отличается от обучения модели?


### Загружаем результат лабораторной 02

Используем `cars_processed.csv`, а не таблицу со средними ценами групп. Положите файл рядом с ноутбуком; без локального файла используется пример из репозитория курса. Код проверяет наличие пропусков и недопустимых для примера значений, но не выполняет новую скрытую очистку.


In [ ]:
data_path = Path('cars_processed.csv')
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(
        'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/'
        'refs/heads/main/Workshops/cars_processed.csv'
    )
df = df.rename(columns={'Price(euro)': 'Price', 'Engine_capacity(cm3)': 'Engine'})
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed:')])
df = df.reset_index(drop=True)
NUMERIC_FEATURES = ['Year', 'Distance', 'Engine', 'km_year']
if df[NUMERIC_FEATURES + ['Price']].isna().any().any():
    raise ValueError('В выбранных колонках есть пропуски: вернитесь к подготовке данных.')
if not np.isfinite(df[NUMERIC_FEATURES + ['Price']].to_numpy()).all():
    raise ValueError('В выбранных колонках есть бесконечные значения.')
if not (df['Price'] > 0).all():
    raise ValueError('Для этого учебного примера нужны положительные цены.')
df.head(3)


Таблица загружена в `df`. Цена измеряется в евро, пробег — в км, объём двигателя — в см³. `Year` — год выпуска.

`km_year` из работы 02 вычислен относительно условного опорного года 2022. Он не измеряет историю пробега. Все результаты относятся к подготовленной учебной области анализа, а не ко всем возможным объявлениям.

**Остановитесь:** сколько строк видно в `head`, и сколько строк реально хранится в таблице?


## 2. X — вопросы модели, y — известные ответы

Начнём с четырёх численных признаков. Марки и модели пока не используем: текст надо отдельно кодировать. `Age` не добавляем вместе с `Year`, поскольку при фиксированном опорном годе они полностью связаны.

Выбираем признаки **по именам**, а не командой «всё кроме цены»: в таблице могут оказаться технические номера строк и средние, вычисленные из цены. Они не должны случайно попадать в модель.


### Выбираем входы и ответ

`X` — таблица с несколькими колонками. `y` — Series с одной ценой для каждой строки. Их индексы должны совпадать.


In [ ]:
X = df[NUMERIC_FEATURES].copy()
y = df['Price'].copy()

display(X.head(3))
display(y.head(3))


В `X` нет колонки `Price`. В `y` нет признаков. Строка с тем же индексом относится к тому же автомобилю.

**Остановитесь:** где находится цена автомобиля с индексом 0?


### Проверяем размеры

Для `X` форма `(n, p)` означает n объектов и p признаков. Для `y` форма `(n,)` означает n ответов.


In [ ]:
print('X:', X.shape)
print('y:', y.shape)
print('Индексы совпадают:', X.index.equals(y.index))


Число строк должно совпасть. Не перемешивайте признаки и ответы отдельно: иначе модель увидит чужую цену рядом с автомобилем.

**Мини-задание:** покажите в одной маленькой таблице признаки и цену первых трёх объектов. Объясните, почему такая таблица удобна для просмотра, но целиком не должна передаваться как `X`.


In [ ]:
# Ваш код для мини-задания: используйте X.head(3).join(y.head(3)).


## 3. Учим на одной части, проверяем на другой

Если модель увидела цену во время обучения, удачное предсказание этой же цены ещё не показывает качество на новой записи. Поэтому выделим три непересекающиеся части:

| Часть | Примерная доля | Для чего используется |
|---|---:|---|
| `train` | 60% | Обучение моделей и преобразований |
| `valid` | 20% | Учебная проверка и сравнение вариантов |
| `test` | 20% | Итоговая проверка после выбора в работе 06 |

В этой работе **тестовые метрики не считаем**. Учимся читать результаты на валидации. Повторный подбор по валидации возможен, но чем больше вариантов проверено, тем больше риск подстроиться под неё.


### Первое разделение

`train_test_split` принимает `X` и `y` вместе и сохраняет соответствие строк. `test_size=0.4` означает 40% во временную отложенную часть. Остальные 60% идут в обучение.

Имена четырёх результатов задаём слева: признаки обучения, признаки отложенной части, ответы обучения, ответы отложенной части. `random_state=42` фиксирует случайный выбор; число 42 не улучшает качество.


In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.4, random_state=42
)
print('Обучение:', X_train.shape, y_train.shape)
print('Отложенная часть:', X_holdout.shape, y_holdout.shape)


Типы сохраняются: для входных Pandas-таблицы и Series мы получили Pandas-таблицы и Series, а не обязательно массивы NumPy.

**Остановитесь:** сколько признаков осталось после разделения? Почему разделение меняет число строк, но не число колонок?


### Второе разделение

Делим временную отложенную часть пополам: 20% исходных строк в валидацию и 20% в тест. Новое фиксированное случайное состояние обеспечивает воспроизводимость этого шага.


In [ ]:
X_valid, X_test, y_valid, y_test = train_test_split(
    X_holdout, y_holdout, test_size=0.5, random_state=43
)
split_sizes = pd.Series({'train': len(X_train), 'valid': len(X_valid), 'test': len(X_test)})
display(split_sizes.to_frame('Число объектов'))
print('Частей в сумме:', split_sizes.sum(), 'Исходных строк:', len(X))


Количество строк в сумме равно исходному. Размеры могут слегка отличаться от точных процентов из-за округления.

Это случайное разделение строк одной выборки. Оно не проверяет перенос на будущий рынок: дат объявлений здесь нет. Повторные объявления одного автомобиля могут сделать оценку слишком оптимистичной.


### Проверяем отсутствие пересечения

Посмотрим на индексы строк. `intersection` возвращает общие метки. Для непересекающихся частей число общих меток равно нулю.


In [ ]:
print('train ∩ valid:', len(X_train.index.intersection(X_valid.index)))
print('train ∩ test:', len(X_train.index.intersection(X_test.index)))
print('valid ∩ test:', len(X_valid.index.intersection(X_test.index)))


Нулевое пересечение индексов подтверждает разделение строк, но не доказывает отсутствие скрытых повторных объявлений.

**Мини-задание:** своими словами объясните, что нельзя делать с `y_test` во время выбора модели.


## 4. До сложной модели: один ответ для всех автомобилей

Начнём с ориентира: всем автомобилям предсказываем медиану цены **обучающей части**. Такой прогноз не учитывает признаки. Он нужен, чтобы понять, даёт ли модель пользу по сравнению с очень простым решением.

`DummyRegressor` — модель-ориентир в scikit-learn. `strategy='median'` выбирает медиану. На обучающих данных медиана минимизирует сумму абсолютных отклонений для постоянного прогноза.


### Создаём модель-ориентир

Первая строка импортирует класс. Вторая создаёт объект с выбранным параметром. Создание объекта ещё не использует данные.


In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy='median')
baseline


`baseline` — объект модели. Он пока не знает медианную цену.

**Остановитесь:** на каком следующем шаге ему станут известны обучающие ответы?


### fit: обучаем

`fit(X_train, y_train)` передаёт модели обучающие признаки и ответы. Здесь модель запоминает медиану `y_train`. Другие модели будут настраивать коэффициенты или правила.


In [ ]:
baseline.fit(X_train, y_train)
print('Медиана обучающих цен:', y_train.median())
print('Значение в модели:', baseline.constant_)


Оба значения совпадают. Символ `_` в конце `constant_` обозначает атрибут, полученный при обучении. Такую запись встретите и у других моделей.

На валидации модель ещё не обучалась.


### predict: получаем оценки

В `predict` передаются **только признаки**. Модель не получает `y_valid`. Для каждого автомобиля вернётся одно предсказанное число.


In [ ]:
baseline_valid_pred = baseline.predict(X_valid)
print('Предсказаний:', baseline_valid_pred.shape)
print('Первые пять:', baseline_valid_pred[:5])


Все числа одинаковые, поскольку это постоянный прогноз. Результат — массив NumPy без индекса Pandas, но в том же порядке, в каком поданы строки `X_valid`.

**Остановитесь:** почему в аргументах `predict` нет цен?


## 5. Как измерить ошибку в евро

Для одного автомобиля ошибка с направлением — $e_i=\hat y_i-y_i$. Положительная означает завышение, отрицательная — занижение. Абсолютная ошибка $|e_i|$ измеряет величину без направления.

MAE — средняя абсолютная ошибка:

$$\mathrm{MAE}=\frac{1}{N}\sum_{i=1}^{N}|\hat y_i-y_i|.$$

MAE = 2000 евро означает среднюю величину отклонения 2000 евро **на этих проверяемых строках**. Это не обещание ошибки не более 2000 евро для каждого автомобиля и не среднее смещение.


### Сначала считаем руками на трёх автомобилях

В маленьком примере есть известные цены и три прогноза. Посмотрим, как ошибки превращаются в одну метрику.


In [ ]:
toy_true = np.array([10_000, 20_000, 30_000])
toy_pred = np.array([12_000, 17_000, 31_000])
toy_error = toy_pred - toy_true
display(pd.DataFrame({'Цена': toy_true, 'Прогноз': toy_pred,
                      'Ошибка': toy_error, 'Абсолютная ошибка': np.abs(toy_error)}))
print('Средняя ошибка с направлением:', toy_error.mean())
print('MAE:', np.abs(toy_error).mean())


Средняя ошибка с направлением равна нулю: завышения и занижения компенсировались. MAE при этом равна 2000 евро. Поэтому MAE нельзя называть смещением среднего.

**Мини-задание:** объясните, почему нулевая средняя ошибка здесь не означает идеальные прогнозы.


### Теперь считаем MAE модели-ориентира

Функция принимает известные ответы первым аргументом, прогнозы — вторым. Она не обучает модель.


In [ ]:
from sklearn.metrics import mean_absolute_error

baseline_valid_mae = mean_absolute_error(y_valid, baseline_valid_pred)
print(f'MAE ориентира на валидации: {baseline_valid_mae:.1f} евро')


Получили первую точку сравнения. Следующая модель должна сравниваться на **тех же** `X_valid` и `y_valid`.

Запишите число и краткую интерпретацию. Не называйте его процентом: MAE здесь в евро.


## 6. Линейная регрессия сначала по одному признаку

Пусть цена оценивается по году выпуска:

$$\hat y = b + w\cdot Year.$$

`w` — коэффициент, `b` — свободный член. `LinearRegression` подбирает их по обучающей части методом наименьших квадратов. Минимизируется сумма квадратов ошибок, а не MAE.

Мы начинаем с одного признака ради понимания. Год выпуска не описывает марку, модель, состояние и комплектацию.


### Подготовим таблицы с одним признаком

Двойные скобки сохраняют двумерную таблицу `(число автомобилей, 1)`, которую ожидает модель.


In [ ]:
X_train_year = X_train[['Year']]
X_valid_year = X_valid[['Year']]
print(X_train_year.shape)
display(X_train_year.head(3))


Входов стало меньше, но список автомобилей и их ответы остались теми же.

**Остановитесь:** чем `X_train['Year']` отличается по форме от `X_train[['Year']]`?


### Создаём объект линейной регрессии


In [ ]:
from sklearn.linear_model import LinearRegression

year_model = LinearRegression()
year_model


Модель создана. Коэффициенты по данным ещё не найдены.


### Обучаем и смотрим параметры


In [ ]:
year_model.fit(X_train_year, y_train)
print('Коэффициент w:', year_model.coef_[0])
print('Свободный член b:', year_model.intercept_)


`coef_` содержит коэффициенты, `intercept_` — свободный член. Их значения зависят от обучающего набора.

Коэффициент показывает изменение **предсказания модели** при увеличении года на единицу. Он не доказывает, что выпуск на год позднее причинно меняет цену на эту величину. Свободный член здесь соответствует условному `Year=0`, вне области данных.


### Предсказываем и сравниваем с ориентиром


In [ ]:
year_valid_pred = year_model.predict(X_valid_year)
year_valid_mae = mean_absolute_error(y_valid, year_valid_pred)
print(f'MAE по одному году: {year_valid_mae:.1f} евро')
print(f'MAE ориентира: {baseline_valid_mae:.1f} евро')


Если MAE уменьшилась, модель дала более точные оценки в среднем по этой части. Если выросла, даже простая медиана оказалась лучше по выбранному критерию. Вывод нужно делать из полученных чисел.


### Посмотрим на одну строку

`iloc[[0]]` сохраняет двумерную таблицу даже для одного автомобиля. Выбираем первый автомобиль в валидации, а не первый в исходном файле.


In [ ]:
one_car = X_valid_year.iloc[[0]]
one_prediction = year_model.predict(one_car)[0]
one_true_price = y_valid.iloc[0]
display(one_car)
print(f'Известная цена: {one_true_price:.1f} евро')
print(f'Прогноз: {one_prediction:.1f} евро')
print(f'Абсолютная ошибка: {abs(one_prediction - one_true_price):.1f} евро')


Ошибка одного объекта может быть намного больше или меньше MAE всей части. Модель здесь не получает известную цену: мы показываем её уже после предсказания для проверки.

**Мини-задание:** повторите просмотр для второго и третьего автомобилей. Не переобучайте модель ради отдельных ответов.


In [ ]:
# Ваш код: используйте X_valid_year.iloc[[1]] и X_valid_year.iloc[[2]].


## 7. Четыре признака и StandardScaler

Теперь используем все четыре численных признака. Они измеряются в разных единицах: год, километры и см³. У многих алгоритмов масштаб влияет на расстояния, регуляризацию и численные вычисления.

Для обычной линейной регрессии без регуляризации само изменение единиц не должно существенно менять прогнозы при корректном решении. Масштабирование изучаем как отдельный полезный инструмент; особенно оно понадобится ближайшим соседям в работе 06.

`StandardScaler` вычитает обучающее среднее и делит на обучающее стандартное отклонение:

$$z=(x-\mu_{train})/s_{train}.$$

Он не делает распределение нормальным, не ограничивает значения интервалом [0, 1] и не требует исходной нормальности. Значения могут быть отрицательными или больше единицы.


### Маленький пример до полной таблицы

Для пробегов 10 000, 20 000 и 30 000 км найдём преобразование. Сначала создаём объект, потом учим параметры.


In [ ]:
from sklearn.preprocessing import StandardScaler

toy_scaler = StandardScaler()
toy_distance = pd.DataFrame({'Distance': [10_000, 20_000, 30_000]})
toy_scaler.fit(toy_distance)
print('Среднее:', toy_scaler.mean_)
print('Стандартное отклонение:', toy_scaler.scale_)
print('Преобразованные значения:', toy_scaler.transform(toy_distance).ravel())


Среднее стало нулевым на обучающих трёх значениях, но исходные значения не превратились в проценты. Scaler использует стандартное отклонение с `ddof=0`; это отличается от `Series.std()` по умолчанию.

**Остановитесь:** что получится у значения, равного обучающему среднему?


### fit для scaler только на train

Scaler тоже обучается: ему надо узнать средние и стандартные отклонения. Ответы `y_train` здесь не нужны, потому что масштабируются входные признаки.


In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
display(pd.DataFrame({'Признак': NUMERIC_FEATURES,
                      'Среднее train': scaler.mean_, 'Масштаб train': scaler.scale_}))


Параметры найдены на обучающих строках. Если вычислить их на всей таблице, проверочная часть повлияет на подготовку модели — возникнет утечка данных.


### transform для train и valid

Один обученный scaler применяем к обеим частям. На валидации не вызываем `fit` повторно.


In [ ]:
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

display(pd.DataFrame(X_train_scaled, columns=NUMERIC_FEATURES, index=X_train.index).head(3))
print('Train:', X_train_scaled.shape, 'Valid:', X_valid_scaled.shape)


Результат `transform` здесь — массив NumPy. Поэтому для просмотра явно вернули имена колонок и индексы.

`fit_transform(X_train)` — сокращение последовательности `fit(X_train)`, затем `transform(X_train)`. На валидации используется только `transform(X_valid)`.

**Мини-задание:** почему нельзя отдельно обучить второй scaler на `X_valid`?


### Новая линейная модель по четырём признакам

Пока не прячем шаги в функцию: видим обучение и предсказание отдельно.


In [ ]:
numeric_model = LinearRegression()
numeric_model.fit(X_train_scaled, y_train)
numeric_valid_pred = numeric_model.predict(X_valid_scaled)
numeric_train_pred = numeric_model.predict(X_train_scaled)
print('Число коэффициентов:', len(numeric_model.coef_))


Для четырёх входных признаков найдено четыре коэффициента и свободный член. Предсказания цены остаются в евро: `y` мы не масштабировали.


## 8. Сравнение, таблица ошибок и график

RMSE — корень из средней квадратичной ошибки:

$$\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2}.$$

MSE — выражение **до** извлечения корня, с единицами евро². RMSE снова измеряется в евро и сильнее реагирует на крупные ошибки. Ни MAE, ни RMSE сами по себе не равны смещению или дисперсии ошибок.


### Оценим четыре признака на train и valid

Здесь `np.sqrt` берёт корень, а `mean_squared_error` считает MSE. Проверяем отдельные части, не объединяя их.


In [ ]:
from sklearn.metrics import mean_squared_error

train_mae = mean_absolute_error(y_train, numeric_train_pred)
valid_mae = mean_absolute_error(y_valid, numeric_valid_pred)
valid_rmse = np.sqrt(mean_squared_error(y_valid, numeric_valid_pred))
print(f'Train MAE: {train_mae:.1f} евро')
print(f'Valid MAE: {valid_mae:.1f} евро')
print(f'Valid RMSE: {valid_rmse:.1f} евро')


Малая обучающая ошибка при большой проверочной может указывать на переобучение. Большие ошибки на обеих частях могут означать недостаточную модель, слабые признаки или неоднородные данные. Один произвольный порог разницы не доказывает ни хорошее обобщение, ни переобучение.


### Сведём три варианта в таблицу

У всех вариантов одна валидация. Базовый прогноз оценивается по тем же строкам.


In [ ]:
comparison_05 = pd.DataFrame({
    'Модель': ['Медиана train', 'Линейная: Year', 'Линейная: четыре признака'],
    'Valid MAE, евро': [baseline_valid_mae, year_valid_mae, valid_mae]
})
display(comparison_05.round(1))


Ответьте по своей таблице: какая модель лучше по MAE? Насколько уменьшилась ошибка относительно медианы? Добавление признаков не обязано улучшать любой результат.


### Посмотрим известные цены и прогнозы вместе

Индекс валидации явно передаём таблице, чтобы цена и прогноз относились к одному автомобилю. Ошибка со знаком и абсолютная ошибка — разные колонки.


In [ ]:
valid_predictions = X_valid.copy()
valid_predictions['Price'] = y_valid
valid_predictions['Prediction'] = numeric_valid_pred
valid_predictions['Error'] = valid_predictions['Prediction'] - valid_predictions['Price']
valid_predictions['Abs_error'] = valid_predictions['Error'].abs()
display(valid_predictions.head(10))


Положительный `Error` означает завышение оценки, отрицательный — занижение. `Abs_error` всегда неотрицателен.

**Мини-задание:** выведите пять наибольших `Abs_error`. Большая ошибка модели ещё не доказывает, что цена в исходной строке неверна.


### График «известная цена — прогноз»

Каждая точка — одна валидационная запись. На пунктирной линии прогноз совпадает с ценой. Выше линии модель завышает, ниже — занижает.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_valid, numeric_valid_pred, s=10, alpha=0.25)
low = min(y_valid.min(), numeric_valid_pred.min())
high = max(y_valid.max(), numeric_valid_pred.max())
ax.plot([low, high], [low, high], 'r--', label='Совпадение')
ax.set(xlabel='Указанная цена, евро', ylabel='Прогноз, евро',
       title='Четыре численных признака: валидация')
ax.legend()
ax.grid(alpha=.3)
plt.show()


Отметьте область с большими ошибками. Линейная модель может выдать отрицательную цену: это показывает ограничение её формы. Не обрезайте такие прогнозы незаметно — это уже отдельное изменение алгоритма, которое нужно оценивать на валидации.


## 9. Дополнительно: MAPE, R² и score

MAPE — средняя абсолютная процентная ошибка. В scikit-learn она возвращается как доля: умножаем на 100 для процентов. При нулевых ценах интерпретация нарушается, а низкие цены получают большой относительный вес.

R² сравнивает сумму квадратов ошибок с отклонениями от средней **цены оцениваемой части**. 1 — идеальное совпадение; 0 — результат уровня постоянной средней этой части по квадратичной ошибке; отрицательное значение — хуже такого ориентира. Это не доля правильных цен.

`score` зависит от класса: у `LinearRegression` это R², у большинства классификаторов — доля правильных ответов. Для понятного отчёта всегда называйте метрику явно.


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score

print(f'Valid MAPE: {100 * mean_absolute_percentage_error(y_valid, numeric_valid_pred):.1f}%')
print(f'Valid R²: {r2_score(y_valid, numeric_valid_pred):.3f}')
print(f'LinearRegression.score: {numeric_model.score(X_valid_scaled, y_valid):.3f}')


## 10. Самостоятельная работа и передача в 06

1. Объясните `X`, `y`, `fit`, `predict` на примере одного автомобиля.
2. Выпишите размеры трёх частей и их назначение.
3. Сравните медиану и обе линейные модели по MAE одной валидации.
4. Проверьте пять больших ошибок: покажите признаки, цену, прогноз и направление ошибки.
5. *Дополнительно:* сравните линейную модель с масштабированием и без него на тех же строках. Объясните, почему это отличается от добавления нового признака.

**Что сдавать:** выполненный ноутбук, таблицы сравнения и валидационных прогнозов; краткий вывод из 5–7 предложений. Тестовые метрики пока не вычисляйте.

**Оценивание, 10 баллов:** постановка и входы — 2; разделение — 2; обучение и предсказание — 2; метрики и график — 2; интерпретация и вывод — 2.

Мы обучили простой ориентир и линейные модели, получили цены в евро и проверили ошибки на отложенной валидации. В работе 06 сравним более гибкие варианты по той же схеме, выберем модель и выполним итоговую тестовую проверку.


### Сохранение результатов

Записываем таблицы без индекса в CSV; номер исходной рабочей строки сохраняем отдельной колонкой. Не создаём признаков из известных цен.


In [ ]:
comparison_05.to_csv('lab05_model_comparison.csv', index=False)
valid_predictions.rename_axis('dataset_row_id').reset_index().to_csv(
    'lab05_valid_predictions.csv', index=False
)
split_manifest = pd.DataFrame({'dataset_row_id': df.index, 'split': ''})
split_manifest.loc[X_train.index, 'split'] = 'train'
split_manifest.loc[X_valid.index, 'split'] = 'valid'
split_manifest.loc[X_test.index, 'split'] = 'test'
split_manifest.to_csv('lab05_split_manifest.csv', index=False)


Работа 06 повторяет тот же отбор строк при том же файле, его порядке и случайных состояниях. CSV с распределением строк позволяет проверить совпадение. Если меняется файл, его порядок или состав, это уже другое разделение.


Документация: [первые шаги scikit-learn](https://scikit-learn.org/stable/getting_started.html), [ошибки подготовки данных](https://scikit-learn.org/stable/common_pitfalls.html), [метрики регрессии](https://scikit-learn.org/stable/modules/model_evaluation.html).
